# AI Lab 5: Local Search
## Hill Climbing, Simulated Annealing, Local Beam Search, and Genetic Algorithm

**Name:** Laiba  
**ID:** 23549009041

This notebook implements four local search algorithms for the 6-Queens problem and a Bangladesh delivery route optimization problem.


## Objective

The objective of this lab is to understand how local search algorithms find good solutions by improving candidate states. Hill Climbing, Simulated Annealing, Local Beam Search, and Genetic Algorithm are applied to two optimization problems and their results are compared.


## Level 1: Toy Graph / 6-Queens

The state is a list of six numbers, where each number represents the row of a queen in one column. The cost is the number of attacking pairs of queens. The goal is to reach cost 0.


In [1]:
# Cell 1: Import required libraries
import random
import math

random.seed(42)
print("Laiba 23549009041")


Laiba 23549009041


In [2]:
# Cell 2: Basic setup for the 6-Queens problem
N = 6
start_state = [0, 0, 0, 0, 0, 0]

print("Starting state:", start_state)
print("Number of queens:", N)


Starting state: [0, 0, 0, 0, 0, 0]
Number of queens: 6


In [3]:
# Cell 3: Generate a random 6-Queens state
def random_state():
    return [random.randint(0, N - 1) for _ in range(N)]

print("Random state:", random_state())


Random state: [5, 0, 0, 5, 2, 1]


In [4]:
# Cell 4: Cost function for the 6-Queens problem
def cost(state):
    conflicts = 0
    for i in range(N):
        for j in range(i + 1, N):
            same_row = state[i] == state[j]
            same_diagonal = abs(state[i] - state[j]) == abs(i - j)
            if same_row or same_diagonal:
                conflicts += 1
    return conflicts

print("Cost of starting state:", cost(start_state))


Cost of starting state: 15


In [5]:
# Cell 5: Generate all 30 neighbors
def get_neighbors(state):
    neighbors = []
    for col in range(N):
        for row in range(N):
            if row != state[col]:
                neighbor = list(state)
                neighbor[col] = row
                neighbors.append(neighbor)
    return neighbors

neighbors = get_neighbors(start_state)
print("Number of neighbors:", len(neighbors))
print("First 5 neighbors:", neighbors[:5])


Number of neighbors: 30
First 5 neighbors: [[1, 0, 0, 0, 0, 0], [2, 0, 0, 0, 0, 0], [3, 0, 0, 0, 0, 0], [4, 0, 0, 0, 0, 0], [5, 0, 0, 0, 0, 0]]


In [6]:
# Cell 6: Hill Climbing
def hill_climbing(start_state):
    current = list(start_state)
    steps = 0

    while True:
        current_cost = cost(current)
        neighbors = get_neighbors(current)
        best_neighbor = min(neighbors, key=cost)
        best_cost = cost(best_neighbor)

        if best_cost < current_cost:
            current = best_neighbor
            steps += 1
        else:
            break

    return current, cost(current), steps

hc_state, hc_cost, hc_steps = hill_climbing(start_state)

print("Hill Climbing result")
print("Final state:", hc_state)
print("Final cost:", hc_cost)
print("Steps:", hc_steps)


Hill Climbing result
Final state: [1, 5, 2, 0, 3, 0]
Final cost: 1
Steps: 4


In [7]:
# Cell 7: Simulated Annealing
def simulated_annealing(start_state, initial_temp=20.0,
                        cooling_rate=0.98, iterations=2000):
    current = list(start_state)
    best = list(current)
    temperature = initial_temp

    for step in range(iterations):
        if temperature <= 1e-6:
            break

        neighbor = random.choice(get_neighbors(current))
        current_cost = cost(current)
        neighbor_cost = cost(neighbor)
        delta = neighbor_cost - current_cost

        if delta < 0:
            current = neighbor
        else:
            probability = math.exp(-delta / temperature)
            if random.random() < probability:
                current = neighbor

        if cost(current) < cost(best):
            best = list(current)

        if cost(best) == 0:
            break

        temperature *= cooling_rate

    return best, cost(best), step + 1

sa_state, sa_cost, sa_steps = simulated_annealing(start_state)

print("Simulated Annealing result")
print("Best state:", sa_state)
print("Best cost:", sa_cost)
print("Steps:", sa_steps)


Simulated Annealing result
Best state: [5, 1, 5, 0, 2, 4]
Best cost: 1
Steps: 834


In [8]:
# Cell 8: Local Beam Search
def local_beam_search(start_state, k=8, max_steps=200):
    states = [list(start_state)]
    best_state = list(start_state)
    best_cost = cost(best_state)

    for step in range(max_steps):
        all_candidates = []

        for state in states:
            all_candidates.extend(get_neighbors(state))

        unique = {}
        for state in all_candidates:
            unique[tuple(state)] = state

        candidates = list(unique.values())
        candidates.sort(key=cost)

        if not candidates:
            break

        states = candidates[:k]

        if cost(states[0]) < best_cost:
            best_state = list(states[0])
            best_cost = cost(states[0])

        if best_cost == 0:
            return best_state, best_cost, step + 1

        if all(cost(s) >= best_cost for s in states):
            return best_state, best_cost, step + 1

    return best_state, best_cost, step + 1

beam_state, beam_cost, beam_steps = local_beam_search(start_state, k=8)

print("Local Beam Search result")
print("Best state:", beam_state)
print("Best cost:", beam_cost)
print("Steps:", beam_steps)


Local Beam Search result
Best state: [0, 5, 0, 0, 0, 0]
Best cost: 10
Steps: 1


In [9]:
# Cell 9: Genetic Algorithm
def single_point_crossover(parent1, parent2):
    point = random.randint(1, N - 1)
    return parent1[:point] + parent2[point:]

def mutation(state, mutation_rate=0.10):
    child = list(state)
    if random.random() < mutation_rate:
        col = random.randint(0, N - 1)
        child[col] = random.randint(0, N - 1)
    return child

def genetic_algorithm(start_state, population_size=30,
                      generations=500, mutation_rate=0.10):
    population = [list(start_state)]
    while len(population) < population_size:
        population.append(random_state())

    best_state = min(population, key=cost)
    best_cost = cost(best_state)

    for generation in range(generations):
        population.sort(key=cost)

        if cost(population[0]) < best_cost:
            best_state = list(population[0])
            best_cost = cost(population[0])

        if best_cost == 0:
            break

        elites = population[:4]
        new_population = [list(state) for state in elites]

        while len(new_population) < population_size:
            parent1 = random.choice(population[:15])
            parent2 = random.choice(population[:15])
            child = single_point_crossover(parent1, parent2)
            child = mutation(child, mutation_rate)
            new_population.append(child)

        population = new_population

    return best_state, best_cost, generation + 1

ga_state, ga_cost, ga_generations = genetic_algorithm(start_state)

print("Genetic Algorithm result")
print("Best state:", ga_state)
print("Best cost:", ga_cost)
print("Generations:", ga_generations)


Genetic Algorithm result
Best state: [3, 5, 0, 2, 4, 1]
Best cost: 1
Generations: 500


In [10]:
# Cell 10: Compare Level 1 results
print("LEVEL 1 RESULTS")
print("-" * 50)
print("Hill Climbing       :", hc_state, "Cost =", hc_cost, "Steps =", hc_steps)
print("Simulated Annealing :", sa_state, "Cost =", sa_cost, "Steps =", sa_steps)
print("Local Beam Search   :", beam_state, "Cost =", beam_cost, "Steps =", beam_steps)
print("Genetic Algorithm   :", ga_state, "Cost =", ga_cost, "Generations =", ga_generations)


LEVEL 1 RESULTS
--------------------------------------------------
Hill Climbing       : [1, 5, 2, 0, 3, 0] Cost = 1 Steps = 4
Simulated Annealing : [5, 1, 5, 0, 2, 4] Cost = 1 Steps = 834
Local Beam Search   : [0, 5, 0, 0, 0, 0] Cost = 10 Steps = 1
Genetic Algorithm   : [3, 5, 0, 2, 4, 1] Cost = 1 Generations = 500


## Level 1 Written Question

Plain Hill Climbing can get stuck at a local optimum because it only accepts a better neighboring state. It cannot move temporarily to a worse state, even when that move may lead to a better solution later. Simulated Annealing sometimes accepts worse moves, Local Beam Search keeps several candidates, and Genetic Algorithm explores a population using crossover and mutation. These methods try to reduce the chance of getting trapped in a poor local solution.


## Level 2: Bangladesh Delivery Route Optimization

A state is a permutation of the eight cities. The route is a closed loop, so the distance from the last city back to the first city is included. All four algorithms use the same adjacent-swap neighbor function, which gives eight neighbors for each route.


In [11]:
# Cell 11: Cities and distance matrix
cities = [
    'Rangpur', 'Bogura', 'Rajshahi', 'Dhaka',
    'Mymensingh', 'Comilla', 'Chittagong', "Cox's Bazar"
]

dist_matrix = {
    'Rangpur': {'Rangpur':0,'Bogura':60,'Rajshahi':125,'Dhaka':270,'Mymensingh':190,'Comilla':370,'Chittagong':520,"Cox's Bazar":670},
    'Bogura': {'Rangpur':60,'Bogura':0,'Rajshahi':65,'Dhaka':210,'Mymensingh':250,'Comilla':310,'Chittagong':460,"Cox's Bazar":610},
    'Rajshahi': {'Rangpur':125,'Bogura':65,'Rajshahi':0,'Dhaka':275,'Mymensingh':315,'Comilla':375,'Chittagong':525,"Cox's Bazar":675},
    'Dhaka': {'Rangpur':270,'Bogura':210,'Rajshahi':275,'Dhaka':0,'Mymensingh':120,'Comilla':100,'Chittagong':250,"Cox's Bazar":400},
    'Mymensingh': {'Rangpur':190,'Bogura':250,'Rajshahi':315,'Dhaka':120,'Mymensingh':0,'Comilla':220,'Chittagong':370,"Cox's Bazar":520},
    'Comilla': {'Rangpur':370,'Bogura':310,'Rajshahi':375,'Dhaka':100,'Mymensingh':220,'Comilla':0,'Chittagong':150,"Cox's Bazar":300},
    'Chittagong': {'Rangpur':520,'Bogura':460,'Rajshahi':525,'Dhaka':250,'Mymensingh':370,'Comilla':150,'Chittagong':0,"Cox's Bazar":150},
    "Cox's Bazar": {'Rangpur':670,'Bogura':610,'Rajshahi':675,'Dhaka':400,'Mymensingh':520,'Comilla':300,'Chittagong':150,"Cox's Bazar":0}
}

start_route = list(cities)

print("Laiba 23549009041")
print("Starting route:", start_route)


Laiba 23549009041
Starting route: ['Rangpur', 'Bogura', 'Rajshahi', 'Dhaka', 'Mymensingh', 'Comilla', 'Chittagong', "Cox's Bazar"]


In [12]:
# Cell 12: Route cost
def route_cost(route):
    total = 0
    n = len(route)
    for i in range(n):
        total += dist_matrix[route[i]][route[(i + 1) % n]]
    return total

print("Starting route cost:", route_cost(start_route))


Starting route cost: 1710


In [13]:
# Cell 13: Generate 8 adjacent-swap neighbors
def route_neighbors(route):
    neighbors = []
    n = len(route)

    for i in range(n):
        j = (i + 1) % n
        neighbor = list(route)
        neighbor[i], neighbor[j] = neighbor[j], neighbor[i]
        neighbors.append(neighbor)

    return neighbors

route_neighbours = route_neighbors(start_route)

print("Number of route neighbors:", len(route_neighbours))
print("First 2 neighbors:")
print(route_neighbours[0])
print(route_neighbours[1])


Number of route neighbors: 8
First 2 neighbors:
['Bogura', 'Rangpur', 'Rajshahi', 'Dhaka', 'Mymensingh', 'Comilla', 'Chittagong', "Cox's Bazar"]
['Rangpur', 'Rajshahi', 'Bogura', 'Dhaka', 'Mymensingh', 'Comilla', 'Chittagong', "Cox's Bazar"]


In [14]:
# Cell 14: Route Hill Climbing
def route_hill_climbing(start_route):
    current = list(start_route)
    steps = 0

    while True:
        current_cost = route_cost(current)
        neighbors = route_neighbors(current)
        best_neighbor = min(neighbors, key=route_cost)
        best_cost = route_cost(best_neighbor)

        if best_cost < current_cost:
            current = best_neighbor
            steps += 1
        else:
            break

    return current, route_cost(current), steps

route_hc, route_hc_cost, route_hc_steps = route_hill_climbing(start_route)

print("Route Hill Climbing result")
print("Tour:", route_hc)
print("Cost:", route_hc_cost)
print("Steps:", route_hc_steps)


Route Hill Climbing result
Tour: ['Rangpur', 'Bogura', 'Rajshahi', 'Mymensingh', 'Dhaka', 'Comilla', 'Chittagong', "Cox's Bazar"]
Cost: 1630
Steps: 1


In [15]:
# Cell 15: Route Simulated Annealing
def route_simulated_annealing(start_route, initial_temp=500.0,
                              cooling_rate=0.995, iterations=5000):
    current = list(start_route)
    best = list(current)
    temperature = initial_temp

    for step in range(iterations):
        if temperature <= 1e-6:
            break

        neighbor = random.choice(route_neighbors(current))
        delta = route_cost(neighbor) - route_cost(current)

        if delta < 0:
            current = neighbor
        else:
            probability = math.exp(-delta / temperature)
            if random.random() < probability:
                current = neighbor

        if route_cost(current) < route_cost(best):
            best = list(current)

        temperature *= cooling_rate

    return best, route_cost(best), step + 1

route_sa, route_sa_cost, route_sa_steps = route_simulated_annealing(start_route)

print("Route Simulated Annealing result")
print("Tour:", route_sa)
print("Cost:", route_sa_cost)
print("Steps:", route_sa_steps)


Route Simulated Annealing result
Tour: ['Mymensingh', 'Comilla', "Cox's Bazar", 'Chittagong', 'Dhaka', 'Bogura', 'Rajshahi', 'Rangpur']
Cost: 1510
Steps: 3998


In [16]:
# Cell 16: Route Local Beam Search
def route_local_beam_search(start_route, k=8, max_steps=300):
    states = [list(start_route)]
    best_state = list(start_route)
    best_cost = route_cost(best_state)

    for step in range(max_steps):
        all_candidates = []

        for state in states:
            all_candidates.extend(route_neighbors(state))

        unique = {}
        for route in all_candidates:
            unique[tuple(route)] = route

        candidates = list(unique.values())
        candidates.sort(key=route_cost)

        if not candidates:
            break

        states = candidates[:k]

        if route_cost(states[0]) < best_cost:
            best_state = list(states[0])
            best_cost = route_cost(states[0])

        if best_cost == 0:
            break

        if all(route_cost(s) >= best_cost for s in states):
            break

    return best_state, best_cost, step + 1

route_beam, route_beam_cost, route_beam_steps = route_local_beam_search(start_route, k=8)

print("Route Local Beam Search result")
print("Tour:", route_beam)
print("Cost:", route_beam_cost)
print("Steps:", route_beam_steps)


Route Local Beam Search result
Tour: ['Rangpur', 'Bogura', 'Rajshahi', 'Mymensingh', 'Dhaka', 'Comilla', 'Chittagong', "Cox's Bazar"]
Cost: 1630
Steps: 1


In [17]:
# Cell 17: Order crossover and route mutation
def order_crossover(parent1, parent2):
    n = len(parent1)
    left, right = sorted(random.sample(range(n), 2))

    child = [None] * n
    child[left:right + 1] = parent1[left:right + 1]

    remaining = [city for city in parent2 if city not in child]

    index = 0
    for i in range(n):
        if child[i] is None:
            child[i] = remaining[index]
            index += 1

    return child

def route_mutation(route, mutation_rate=0.10):
    child = list(route)

    if random.random() < mutation_rate:
        i = random.randint(0, len(child) - 1)
        j = (i + 1) % len(child)
        child[i], child[j] = child[j], child[i]

    return child

test_parent1 = start_route
test_parent2 = list(reversed(start_route))
test_child = order_crossover(test_parent1, test_parent2)

print("Parent 1:", test_parent1)
print("Parent 2:", test_parent2)
print("Child:", test_child)
print("Child is a valid tour:", len(test_child) == len(set(test_child)) == len(cities))


Parent 1: ['Rangpur', 'Bogura', 'Rajshahi', 'Dhaka', 'Mymensingh', 'Comilla', 'Chittagong', "Cox's Bazar"]
Parent 2: ["Cox's Bazar", 'Chittagong', 'Comilla', 'Mymensingh', 'Dhaka', 'Rajshahi', 'Bogura', 'Rangpur']
Child: ["Cox's Bazar", 'Chittagong', 'Rajshahi', 'Dhaka', 'Mymensingh', 'Comilla', 'Bogura', 'Rangpur']
Child is a valid tour: True


In [18]:
# Cell 18: Route Genetic Algorithm
def route_genetic_algorithm(start_route, population_size=40,
                            generations=600, mutation_rate=0.10):
    population = [list(start_route)]

    while len(population) < population_size:
        route = list(cities)
        random.shuffle(route)
        population.append(route)

    best_state = min(population, key=route_cost)
    best_cost = route_cost(best_state)

    for generation in range(generations):
        population.sort(key=route_cost)

        if route_cost(population[0]) < best_cost:
            best_state = list(population[0])
            best_cost = route_cost(population[0])

        elites = population[:4]
        new_population = [list(route) for route in elites]

        while len(new_population) < population_size:
            parent1 = random.choice(population[:20])
            parent2 = random.choice(population[:20])

            child = order_crossover(parent1, parent2)
            child = route_mutation(child, mutation_rate)
            new_population.append(child)

        population = new_population

    return best_state, best_cost, generation + 1

route_ga, route_ga_cost, route_ga_generations = route_genetic_algorithm(start_route)

print("Route Genetic Algorithm result")
print("Tour:", route_ga)
print("Cost:", route_ga_cost)
print("Generations:", route_ga_generations)


Route Genetic Algorithm result
Tour: ['Mymensingh', 'Comilla', "Cox's Bazar", 'Chittagong', 'Dhaka', 'Rajshahi', 'Bogura', 'Rangpur']
Cost: 1510
Generations: 600


In [19]:
# Cell 19: Compare Level 2 results
print("LEVEL 2 RESULTS")
print("-" * 70)
print("Hill Climbing       :", route_hc, "Cost =", route_hc_cost, "Steps =", route_hc_steps)
print("Simulated Annealing :", route_sa, "Cost =", route_sa_cost, "Steps =", route_sa_steps)
print("Local Beam Search   :", route_beam, "Cost =", route_beam_cost, "Steps =", route_beam_steps)
print("Genetic Algorithm   :", route_ga, "Cost =", route_ga_cost, "Generations =", route_ga_generations)


LEVEL 2 RESULTS
----------------------------------------------------------------------
Hill Climbing       : ['Rangpur', 'Bogura', 'Rajshahi', 'Mymensingh', 'Dhaka', 'Comilla', 'Chittagong', "Cox's Bazar"] Cost = 1630 Steps = 1
Simulated Annealing : ['Mymensingh', 'Comilla', "Cox's Bazar", 'Chittagong', 'Dhaka', 'Bogura', 'Rajshahi', 'Rangpur'] Cost = 1510 Steps = 3998
Local Beam Search   : ['Rangpur', 'Bogura', 'Rajshahi', 'Mymensingh', 'Dhaka', 'Comilla', 'Chittagong', "Cox's Bazar"] Cost = 1630 Steps = 1
Genetic Algorithm   : ['Mymensingh', 'Comilla', "Cox's Bazar", 'Chittagong', 'Dhaka', 'Rajshahi', 'Bogura', 'Rangpur'] Cost = 1510 Generations = 600


## Order Crossover Explanation

Order crossover guarantees a valid tour because it keeps the selected cities from the first parent and fills the remaining positions with only the missing cities from the second parent, so every city appears exactly once.


## Conclusion

In this lab, I learned how different local search algorithms can be used for optimization problems. Hill Climbing is simple but can stop at a local optimum, while the other methods provide more exploration. I also learned how the same search ideas can be adapted from a queen-placement problem to a real route optimization problem.
